# 01 · Phase-1 bases

Every method in the paper adapts the **same frozen phase-1 model** of each unit: phase 1 fits one source
model (seed 1), picks its decision cut on an inner validation split, and freezes it together with a bridged,
target-initialised copy that phase 2 starts from. This notebook regenerates those **34 bases**:

| family | units | work items (one TPE search each) | entrypoint | paper Kaggle jobs |
|---|---|---|---|---|
| synthetic | 24 (`d{0.6,1,1.5,2.5}` x shift `{0.5,1.5,3}` x rotation `{0,1}`) | 8, one per (d, rotation) | `MIL_CREDA_EEG.p1_syn.run(d, rotation)` | `sp1-d{d}-r{r}` (8, plus a re-run `sp1b-d1p5-r0`) |
| ADHD/Control Children (IEEE) | 10 (rotation `r0..r4` x target-test half `h0/h1`) | 5, one per rotation | `MIL_CREDA_EEG.p1_variants.run(rotation, "A")` | `p1A-r{r}` |

**What the paper quotes from this stage** (checked in the last cell):
* `main.tex:370`: "Every method exceeded the first phase (0.576) on average" — mean phase-1 BA of the 24 synthetic
  units on their 120-subject evaluation pools;
* `main.tex:459`, Table `tab:ieee-ranking`: "Phase 1 (source only) & 4.60 & 0.655 & 0.717 & 0.309" — mean BA, AUC
  and kappa of the 10 ADHD bases on their target-test halves.

It also writes a table of the 34 bases (settings, cut and phase-1 metrics). Every later stage reads the bases
from `MIL-CREDA-EEG/Manifests/{oracle_bases,ieee_pool_bases}` by default; `stage01_bases.bases_dir(regenerated=True)`
points it at the folder regenerated here instead.

## Setup
Works locally (repository checkout) and on Kaggle (the repository is cloned into `/kaggle/working`).

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Daprosero/Domain_Adaptation"
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / "src" / "MIL_CREDA_EEG").is_dir()), None)
if root is None and Path("/kaggle").exists():
    root = Path("/kaggle/working/Domain_Adaptation")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mne==1.10.1", "neurodsp==2.2.1"], check=True)
if root is None:
    raise RuntimeError("run this notebook from inside the Domain_Adaptation repository or on Kaggle")
sys.path.insert(0, str(root / "src"))

from MIL_CREDA_EEG import repro
from MIL_CREDA_EEG.reproduce import stage01_bases as S
print("repository:", root)

## Data

* **Synthetic units** are simulated on the fly by `MIL_CREDA_EEG.eeg_sim` from each unit's recorded settings
  (class effect, shift, artifact rate, event fraction, 60 subjects), read from the tracked base in `Manifests/oracle_bases`.
* **ADHD units** read the 121 IEEE recordings of the public Kaggle dataset
  [`daprosero/mi-tdah-dataset`](https://www.kaggle.com/datasets/daprosero/mi-tdah-dataset) (CC0): mounted when the
  dataset is attached to a Kaggle notebook, otherwise downloaded (about 33 MB) with `kagglehub` by `repro.ieee_root()`.

In [ ]:
print("IEEE ADHD folder:", repro.ieee_root())

## Parameters (those of the paper runs)

* TPE search of phase 1 with Optuna, **20 trials**, sampler seed `xpilot.SAMPLER_SEED + rotation`, over
  `spectral_search.SPACE` (`main.tex:326`: lr 1e-4..1e-2, l2 1e-4..1e-1, width {32, 64, 128}, attention temperature
  0.01..1, epochs {50, 100, 200}), scored by the mean of inner BA and AUC; synthetic: searched on the shift-1.5 unit.
* Base refitted on all source training subjects with **seed 1**, cut where the inner-validation BA peaks
  (`xcompare.shared_base`); **4 torch threads**, CPU.
* Source: `run-config.json` of the Kaggle jobs `sp1-*` (`run.kwargs = {"d", "rotation"}`) and `p1A-*`
  (`{"rotation", "variant": "A"}`), with the defaults of `p1_syn.run` / `p1_variants.run` (`trials=20`, `threads=4`).
* Synthetic bases are also evaluated on the unit's 120-subject evaluation pool (`criteria_pool._setup`, as
  `Results/v82-pool24/eval_bases.py` did; no training).

In [ ]:
print("work items:", [S.unit_name(u) for u in S.UNITS])
print("trials:", S.TRIALS, " threads:", S.THREADS, " ADHD variant:", S.VARIANT)

## Cost of the full run

Measured in the paper campaign (Kaggle CPU, 4 threads, `phase1-d*-r*.json` `seconds`): 1695 to 2429 s per synthetic
work item (mean about 30 min), plus about 1.5 min per synthetic base for its evaluation pool; no duration was recorded for the ADHD
jobs, so an ADHD work item is estimated at about 11 min (on a laptop CPU one TPE trial plus both bases took 74 s and
one base about 20 s, so 20 trials take about 10 min). Whole stage on one CPU
account: about **5.5 h** for the 8 synthetic items and **1 h** for the 5 ADHD items, about 6.5 h in total, no GPU.
The paper ran it as 13 Kaggle CPU jobs (`sp1-*`, `p1A-*`) spread over its accounts.

## Run
`SPLIT = None` runs all 13 work items in this account; `SPLIT = (k, n)` runs items k, k+n, k+2n, ... so n accounts
share the stage (for example `(0, 2)` and `(1, 2)`). `REPRO_SMOKE=1` runs only a wiring check: one ADHD rotation
and one synthetic item with a single TPE trial (about 5 min), into `Results/paper/01_phase1_bases/smoke/`.

In [ ]:
SPLIT = None  # or (k, n) to run portion k of n in one Kaggle account
SMOKE = os.environ.get("REPRO_SMOKE") == "1"
OUT = S.stage_dir() / "smoke" if SMOKE else S.regenerated_dir()
print("this run:", [S.unit_name(u) for u in (S.SMOKE_UNITS if SMOKE else repro.portion(S.UNITS, SPLIT))])
print("output:", OUT / "bases")

In [ ]:
records = S.run(split=SPLIT, smoke=SMOKE, out=OUT)

## Merge
When several accounts ran portions, copy each account's `regenerated/bases/` folder (`oracle_bases/` and
`ieee_pool_bases/`) into the same `Results/paper/01_phase1_bases/regenerated/bases/` and run this cell: it summarizes
whatever bases are present into `regenerated/summary/`. The tracked summary of the paper's bases
(`Results/paper/01_phase1_bases/summary/`) is never overwritten. The regenerated `.pt` files total about 15 MB:
keep them out of git.

In [ ]:
regen_summary = S.merge(OUT)
_ = repro.compare(S.compare_regenerated(regen_summary))

The phase-1 fit is not bit-reproducible across CPU platforms and torch builds (the paper ran on Kaggle's
x86 CPU, torch 2.13.0+cpu): a regenerated base can pick another cut and its BA can differ by a few hundredths,
which is why later stages default to the stored bases.

## Render
The table of the 34 bases, from the stored summary only (no training): `tables/phase1_bases.{csv,tex}` and the
paper's numbers in `tables/phase1_paper_numbers.csv`. `render(regen_summary)` draws the same table from a regenerated summary.

In [ ]:
import pandas as pd
paths = S.render()
display(pd.read_csv(paths["csv"]))

## Which bases do the tracked Manifests hold?
The stored summary records the sha256 of each base the paper's phase-2 campaigns read. This cell says whether
`MIL-CREDA-EEG/Manifests/` holds those exact files.

In [ ]:
m = S.manifests_match()
print(len(m["identical"]), "of 34 tracked bases are the paper's;", len(m["different"]), "differ", m["different"][:3])

## Comparison with the paper
From the stored summary (tolerance 0). Pass `regen_summary` to check a regenerated run (tolerance 0.05).

In [ ]:
_ = repro.compare(S.paper_checks())